# Part B: Federated Representation Learning and Clustering

1.实验设置

In [41]:
import copy
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import (
    DataLoader,
    TensorDataset,
    Subset,
    random_split
)
from torchvision.datasets import MNIST

from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

SEED = 42

NUM_CLIENTS = 5
ROUNDS = 10
LOCAL_EPOCHS = 10
TOTAL_EPOCHS = ROUNDS * LOCAL_EPOCHS

BATCH_SIZE = 128
LR = 0.001

HIDDEN_DIM = 128
LATENT_DIM = 16
NUM_CLUSTERS = 5

DEVICE = torch.device("cpu")

torch.manual_seed(SEED)
torch.set_num_threads(4)

2.加载数据，划分数据

In [42]:
project_root = Path.cwd()

if project_root.name == "week4":
    project_root = project_root.parent

data_root = project_root / "data"

MNIST.mirrors = [
    "https://storage.googleapis.com/cvdf-datasets/mnist/"
]

raw_train = MNIST(
    root=data_root,
    train=True,
    download=True
)

raw_test = MNIST(
    root=data_root,
    train=False,
    download=True
)

# 像素值缩放到 [0, 1]
# 图片形状：[样本数, 1, 28, 28]
train_data = TensorDataset(
    raw_train.data.float().unsqueeze(1) / 255.0,
    raw_train.targets
)

test_data = TensorDataset(
    raw_test.data.float().unsqueeze(1) / 255.0,
    raw_test.targets
)

In [43]:
def make_loader(dataset, shuffle=False, seed=SEED,
                batch_size=BATCH_SIZE):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=torch.Generator().manual_seed(seed)
    )

central_loader = make_loader(
    train_data,
    shuffle=True
)

iid_clients = random_split(
    train_data,
    [0.2] * NUM_CLIENTS,
    generator=torch.Generator().manual_seed(SEED)
)

iid_loaders = [
    make_loader(
        client,
        shuffle=True,
        seed=SEED + client_id
    )
    for client_id, client in enumerate(iid_clients)
]

labels = train_data.tensors[1]

non_iid_clients = []

for client_id in range(NUM_CLIENTS):
    digit1 = 2 * client_id
    digit2 = 2 * client_id + 1

    mask = (labels == digit1) | (labels == digit2)
    indices = torch.where(mask)[0].tolist()

    client_data = Subset(train_data, indices)
    non_iid_clients.append(client_data)

non_iid_loaders = [
    make_loader(
        client,
        shuffle=True,
        seed=SEED + client_id
    )
    for client_id, client in enumerate(non_iid_clients)
]

train_eval_loader = make_loader(
    train_data,
    batch_size=1024
)

test_loader = make_loader(
    test_data,
    batch_size=1024
)


print("Training samples:", len(train_data))
print("Test samples:", len(test_data))

print("\nIID client sizes:")
print([len(client) for client in iid_clients])

print("\nnon-IID client sizes:")
print([len(client) for client in non_iid_clients])

Training samples: 60000
Test samples: 10000

IID client sizes:
[12000, 12000, 12000, 12000, 12000]

non-IID client sizes:
[12665, 12089, 11263, 12183, 11800]


3.建立模型


In [44]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, LATENT_DIM)
        )

        self.decoder = nn.Sequential(
            nn.Linear(LATENT_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, 784),
            nn.Sigmoid()
        )

    def forward(self, images):
        features = self.encoder(images)

        reconstructed = self.decoder(features)

        return reconstructed.reshape_as(images)


# 两种训练方式使用相同的初始参数
initial_ae = Autoencoder()

initial_state = copy.deepcopy(
    initial_ae.state_dict()
)


def new_ae():
    model = Autoencoder().to(DEVICE)
    model.load_state_dict(initial_state)
    return model

4.训练模型

In [45]:

# 公共训练函数

def train_ae(model, loader, epochs):
    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR
    )

    criterion = nn.MSELoss()

    for epoch in range(epochs):
        for images, _ in loader:
            images = images.to(DEVICE)

            optimizer.zero_grad()

            reconstructed = model(images)
            loss = criterion(reconstructed, images)

            loss.backward()
            optimizer.step()

In [46]:
#Centralized training
central_ae = new_ae()

train_ae(
    model=central_ae,
    loader=central_loader,
    epochs=TOTAL_EPOCHS
)

print("Centralized AE training completed")

Centralized AE training completed


In [47]:
#Local AE：IID
local_iid_models = []

for client_id, client_loader in enumerate(iid_loaders):
    local_ae = new_ae()

    train_ae(
        model=local_ae,
        loader=client_loader,
        epochs=TOTAL_EPOCHS
    )

    local_iid_models.append(local_ae)

    print(f"Local AE IID: client {client_id} completed")

Local AE IID: client 0 completed
Local AE IID: client 1 completed
Local AE IID: client 2 completed
Local AE IID: client 3 completed
Local AE IID: client 4 completed


In [48]:
#Local AE：non-IID
local_non_iid_models = []

for client_id, client_loader in enumerate(non_iid_loaders):
    local_ae = new_ae()

    train_ae(
        model=local_ae,
        loader=client_loader,
        epochs=TOTAL_EPOCHS
    )

    local_non_iid_models.append(local_ae)

    print(f"Local AE non-IID: client {client_id} completed")

Local AE non-IID: client 0 completed
Local AE non-IID: client 1 completed
Local AE non-IID: client 2 completed
Local AE non-IID: client 3 completed
Local AE non-IID: client 4 completed


In [49]:
#FedAvg AE：IID
iid_global_ae = new_ae()

for round_id in range(1, ROUNDS + 1):

    averaged_state = {
        name: torch.zeros_like(parameter)
        for name, parameter
        in iid_global_ae.state_dict().items()
    }

    total_samples = sum(
        len(loader.dataset)
        for loader in iid_loaders
    )

    for client_loader in iid_loaders:

        # 下发当前全局 AE
        local_ae = copy.deepcopy(iid_global_ae)

        # 本地训练
        train_ae(
            model=local_ae,
            loader=client_loader,
            epochs=LOCAL_EPOCHS
        )

        weight = len(client_loader.dataset) / total_samples

        # 读取本地参数，并加权聚合 Encoder 和 Decoder
        for name, parameter in local_ae.state_dict().items():
            averaged_state[name] += weight * parameter

    iid_global_ae.load_state_dict(averaged_state)

    print(f"FedAvg AE IID: round {round_id} completed")

FedAvg AE IID: round 1 completed
FedAvg AE IID: round 2 completed
FedAvg AE IID: round 3 completed
FedAvg AE IID: round 4 completed
FedAvg AE IID: round 5 completed
FedAvg AE IID: round 6 completed
FedAvg AE IID: round 7 completed
FedAvg AE IID: round 8 completed
FedAvg AE IID: round 9 completed
FedAvg AE IID: round 10 completed


In [50]:
#FedAvg AE：non-IID
non_iid_global_ae = new_ae()

for round_id in range(1, ROUNDS + 1):

    averaged_state = {
        name: torch.zeros_like(parameter)
        for name, parameter
        in non_iid_global_ae.state_dict().items()
    }

    total_samples = sum(
        len(loader.dataset)
        for loader in non_iid_loaders
    )

    for client_loader in non_iid_loaders:

        local_ae = copy.deepcopy(non_iid_global_ae)

        train_ae(
            model=local_ae,
            loader=client_loader,
            epochs=LOCAL_EPOCHS
        )

        weight = len(client_loader.dataset) / total_samples

        for name, parameter in local_ae.state_dict().items():
            averaged_state[name] += weight * parameter

    non_iid_global_ae.load_state_dict(averaged_state)

    print(f"FedAvg AE non-IID: round {round_id} completed")

FedAvg AE non-IID: round 1 completed
FedAvg AE non-IID: round 2 completed
FedAvg AE non-IID: round 3 completed
FedAvg AE non-IID: round 4 completed
FedAvg AE non-IID: round 5 completed
FedAvg AE non-IID: round 6 completed
FedAvg AE non-IID: round 7 completed
FedAvg AE non-IID: round 8 completed
FedAvg AE non-IID: round 9 completed
FedAvg AE non-IID: round 10 completed


5.模型比较

In [51]:
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score
)
SILHOUETTE_MAX_SAMPLES = 2000

#为客户端划分评估数据
iid_eval_sets = random_split(
    test_data,
    [1 / NUM_CLIENTS] * NUM_CLIENTS,
    generator=torch.Generator().manual_seed(SEED + 100)
)

test_labels = test_data.tensors[1]

non_iid_eval_sets = []

for client_id in range(NUM_CLIENTS):
    digit1 = 2 * client_id
    digit2 = 2 * client_id + 1

    mask = (test_labels == digit1) | (test_labels == digit2)
    indices = torch.where(mask)[0].tolist()

    non_iid_eval_sets.append(
        Subset(test_data, indices)
    )

In [52]:
#提取特征和重建 loss
@torch.no_grad()
def extract_features_and_loss(model, loader):
    model.eval()

    all_features = []
    all_labels = []

    squared_error = 0.0
    pixel_count = 0

    for images, labels in loader:
        images = images.to(DEVICE)

        features = model.encoder(images)

        reconstructed = model.decoder(features)
        reconstructed = reconstructed.reshape_as(images)

        all_features.append(features.cpu().numpy())
        all_labels.append(labels.numpy())

        squared_error += (
            (reconstructed - images) ** 2
        ).sum().item()

        pixel_count += images.numel()

    features = np.concatenate(all_features, axis=0)
    labels = np.concatenate(all_labels, axis=0)

    reconstruction_loss = squared_error / pixel_count

    return features, labels, reconstruction_loss

In [53]:
def calculate_silhouette(features, clusters):
    # 统一、可复现地抽取评估样本
    if (
        SILHOUETTE_MAX_SAMPLES is not None
        and len(features) > SILHOUETTE_MAX_SAMPLES
    ):
        rng = np.random.default_rng(SEED)

        indices = rng.choice(
            len(features),
            size=SILHOUETTE_MAX_SAMPLES,
            replace=False
        )

        features = features[indices]
        clusters = clusters[indices]

    number_of_clusters = len(np.unique(clusters))

    # Silhouette 至少需要两个簇，且不能每个样本都独占一个簇
    if not 2 <= number_of_clusters < len(features):
        return np.nan

    return silhouette_score(features, clusters)


In [54]:
#评估一个 AE + K-means
def evaluate_pipeline(model, fit_loader, evaluation_loader):
    # 使用训练特征拟合 K-means，不使用训练标签
    Z_train, _, _ = extract_features_and_loss(
        model,
        fit_loader
    )

    Z_eval, y_eval, reconstruction_loss = (
        extract_features_and_loss(
            model,
            evaluation_loader
        )
    )

    kmeans = KMeans(
        n_clusters=NUM_CLUSTERS,
        n_init=10,
        random_state=SEED
    )

    kmeans.fit(Z_train)

    clusters = kmeans.predict(Z_eval)

    return {
        "ARI": adjusted_rand_score(y_eval, clusters),
        "NMI": normalized_mutual_info_score(y_eval, clusters),
        "Silhouette": calculate_silhouette(Z_eval, clusters),
        "Reconstruction_loss": reconstruction_loss
    }

In [55]:
# Local：客户端指标及其加权平均
METRIC_COLUMNS = [
    "ARI",
    "NMI",
    "Silhouette",
    "Reconstruction_loss"
]


def evaluate_local_models(
    method,
    models,
    client_train_loaders,
    client_eval_sets
):
    assert (
        len(models)
        == len(client_train_loaders)
        == len(client_eval_sets)
    )

    client_records = []

    for client_id, (model, train_loader, eval_set) in enumerate(
        zip(models, client_train_loaders, client_eval_sets)
    ):
        # 本地 K-means 只拟合自己的训练特征
        local_fit_loader = make_loader(
            train_loader.dataset,
            batch_size=1024
        )

        # 本地指标只在自己的评估子集上计算
        local_eval_loader = make_loader(
            eval_set,
            batch_size=1024
        )

        scores = evaluate_pipeline(
            model,
            local_fit_loader,
            local_eval_loader
        )

        client_records.append({
            "method": method,
            "client": client_id,
            "n_train": len(train_loader.dataset),
            "n_eval": len(eval_set),
            **scores,
            "Communication_rounds": 0
        })

    client_metrics = pd.DataFrame(client_records)

    # 按评估样本数加权，而不是简单平均
    weights = (
        client_metrics["n_eval"]
        / client_metrics["n_eval"].sum()
    )

    client_metrics["evaluation_weight"] = weights

    weighted_scores = {
        metric: np.average(
            client_metrics[metric],
            weights=weights
        )
        for metric in METRIC_COLUMNS
    }

    # 如果某客户端的 Silhouette 无法定义，
    # 加权结果保留 NaN，不静默忽略该客户端。
    weighted_scores["Communication_rounds"] = 0

    return client_metrics, weighted_scores


local_iid_details, local_iid_scores = evaluate_local_models(
    method="Local AE + local K-means (IID)",
    models=local_iid_models,
    client_train_loaders=iid_loaders,
    client_eval_sets=iid_eval_sets
)

local_non_iid_details, local_non_iid_scores = (
    evaluate_local_models(
        method="Local AE + local K-means (non-IID)",
        models=local_non_iid_models,
        client_train_loaders=non_iid_loaders,
        client_eval_sets=non_iid_eval_sets
    )
)


In [56]:
#Centralized：完整测试集
central_scores = evaluate_pipeline(
    model=central_ae,
    fit_loader=train_eval_loader,
    evaluation_loader=test_loader
)

central_scores["Communication_rounds"] = 0


In [57]:
#Federated：全局 Encoder + 完整测试集
fed_iid_scores = evaluate_pipeline(
    model=iid_global_ae,
    fit_loader=train_eval_loader,
    evaluation_loader=test_loader
)

fed_iid_scores["Communication_rounds"] = ROUNDS


fed_non_iid_scores = evaluate_pipeline(
    model=non_iid_global_ae,
    fit_loader=train_eval_loader,
    evaluation_loader=test_loader
)

fed_non_iid_scores["Communication_rounds"] = ROUNDS

In [58]:
comparison = pd.DataFrame([
    {
        "method": "Centralized AE + K-means",
        **central_scores
    },
    {
        "method": "Local AE + local K-means (IID, weighted)",
        **local_iid_scores
    },
    {
        "method": "Local AE + local K-means (non-IID, weighted)",
        **local_non_iid_scores
    },
    {
        "method": "Federated AE + global K-means (IID)",
        **fed_iid_scores
    },
    {
        "method": "Federated AE + global K-means (non-IID)",
        **fed_non_iid_scores
    }
])

print("\nFinal comparison:")

print(
    comparison.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)


Final comparison:
                                      method    ARI    NMI  Silhouette  Reconstruction_loss  Communication_rounds
                    Centralized AE + K-means 0.2893 0.4191      0.1372               0.0099                     0
    Local AE + local K-means (IID, weighted) 0.2751 0.4055      0.1313               0.0129                     0
Local AE + local K-means (non-IID, weighted) 0.3834 0.5183      0.1808               0.0091                     0
         Federated AE + global K-means (IID) 0.2938 0.4242      0.1373               0.0115                    10
     Federated AE + global K-means (non-IID) 0.2732 0.3889      0.1298               0.0167                    10


Centralized AE：整体聚类效果最好,一个 AE 连续学习全部类别，不需要合并不同客户端的参数。在这次实验中，它学到的特征更适合按数字类别聚类。但分数仍未接近 1，说明不同数字的特征还存在混杂。

Local IID：各自学习，效果接近集中式，但略弱, 每个客户端虽然都包含 0～9，但训练数据量只有完整数据集的一部分，所以可能难以学到同样充分的特征。

FedAvg IID：重建改善，不保证聚类同步改善
相对于 Local IID：
- 重建 loss 下降
- ARI 却减少
这是因为 AE 训练时只要求“恢复图片”，没有要求“同一种数字必须聚在一起”。模型可能更好地保留了笔画、粗细等细节，却没有让数字类别分得更清楚。

FedAvg non-IID：客户端偏向不同类别，统一学习更困难,各客户端只见到两类数字，本地更新容易偏向自己的数据。服务器平均这些参数后，需要兼顾全部十类，可能出现相互干扰，因此重建和聚类效果都受到影响。

Local non-IID：重建 loss 最低，但聚类指标需要特别解读
它的重建 loss 最低，因为每个 AE 只需要学习并重建自己的两类数字，任务相对简单；这不能直接说明它对全部十类数字的重建能力最强。

What knowledge is shared when AE parameters are aggregated?

Ans:The shared knowledge is how to extract features and reconstruct images, encoded in the AE’s weights and biases.

Why may averaging neural-network parameters be easier than directly averaging local cluster labels?

Ans:Local cluster labels are arbitrary and may have different meanings across clients, so averaging them is not meaningful. Neural networks with the same architecture and initialization have corresponding parameter positions that can be aggregated using FedAvg.

Why do independently learned cluster IDs have a permutation problem?

Ans:Cluster IDs are arbitrary. Different clients may assign different IDs to the same underlying group, so cluster IDs are not aligned across clients.

How does non-IID data affect the learned latent space?

Ans:It makes each client’s latent space more specialized to its local data.

What is the difference between federated representation learning and federated clustering?

Ans:Federated representation learning focuses on learning useful features from distributed data, while federated clustering focuses on grouping distributed samples into clusters.